In [ ]:
from pathlib import Path
import sys
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Find repo root so local package imports work even when cwd is analysis/notebooks
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "quantum_jobs").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Could not find repo root containing quantum_jobs/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quantum_jobs.db.paths import DB_PATH as PROJECT_DB_PATH

db_path = Path(PROJECT_DB_PATH).resolve()
print(f"Using DB: {db_path}")
print(f"Exists: {db_path.exists()}")

if not db_path.exists():
    raise FileNotFoundError(
        f"Expected database does not exist: {db_path}. "
        "Refusing to create a blank SQLite DB."
    )

conn = sqlite3.connect(db_path)


In [ ]:
# Inspect available tables before running analysis queries
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)
display(tables)

table_names = set(tables["name"])
if "job_snapshots" not in table_names:
    raise RuntimeError(
        f"job_snapshots table not found in database: {db_path}\n"
        f"Tables found: {sorted(table_names)}\n"
        "This usually means the notebook is connected to the wrong or blank SQLite DB."
    )


In [ ]:
def run_query(query: str) -> pd.DataFrame:
    return pd.read_sql(query, conn)


## Canonical daily snapshots for longitudinal counts

Raw date-level counts can be inflated when the collector runs multiple times on the same day for the same company.
If we aggregate naively across all rows for a date, we can accidentally sum multiple snapshots together.

To avoid this, this notebook defines **one canonical snapshot per company per date**: the latest `pulled_at` for that `(company, pulled_date)` pair, then counts distinct `job_id` values only inside that snapshot.

For the all-company chart, we start from the first date where at least two companies are present, so the plot focuses on the period where comparisons are meaningful.


In [ ]:
daily_runs_query = """
SELECT
    company,
    COALESCE(pulled_date, substr(pulled_at, 1, 10)) AS snapshot_date,
    COUNT(DISTINCT pulled_at) AS collector_runs,
    COUNT(*) AS raw_rows
FROM job_snapshots
GROUP BY company, snapshot_date
HAVING COUNT(DISTINCT pulled_at) > 1
ORDER BY collector_runs DESC, snapshot_date DESC, company
"""

daily_runs = run_query(daily_runs_query)
daily_runs.head(20)


In [ ]:
canonical_daily_query = """
WITH daily_latest AS (
    SELECT
        company,
        COALESCE(pulled_date, substr(pulled_at, 1, 10)) AS snapshot_date,
        MAX(pulled_at) AS latest_pulled_at
    FROM job_snapshots
    GROUP BY company, snapshot_date
),
canonical_daily_counts AS (
    SELECT
        s.company,
        d.snapshot_date,
        COUNT(DISTINCT s.job_id) AS job_count
    FROM job_snapshots s
    JOIN daily_latest d
      ON s.company = d.company
     AND COALESCE(s.pulled_date, substr(s.pulled_at, 1, 10)) = d.snapshot_date
     AND s.pulled_at = d.latest_pulled_at
    GROUP BY s.company, d.snapshot_date
)
SELECT *
FROM canonical_daily_counts
ORDER BY snapshot_date, company
"""

daily_counts = run_query(canonical_daily_query)
daily_counts["snapshot_date"] = pd.to_datetime(daily_counts["snapshot_date"])
daily_counts.head()


In [ ]:
quantum_machines_sanity_query = """
WITH qm_runs AS (
    SELECT
        COALESCE(pulled_date, substr(pulled_at, 1, 10)) AS snapshot_date,
        pulled_at,
        COUNT(*) AS raw_rows_in_run,
        COUNT(DISTINCT job_id) AS distinct_jobs_in_run
    FROM job_snapshots
    WHERE company = 'Quantum Machines'
    GROUP BY snapshot_date, pulled_at
),
daily_latest AS (
    SELECT
        snapshot_date,
        MAX(pulled_at) AS latest_pulled_at
    FROM qm_runs
    GROUP BY snapshot_date
),
raw_daily AS (
    SELECT
        snapshot_date,
        COUNT(*) AS raw_rows_all_runs
    FROM job_snapshots
    WHERE company = 'Quantum Machines'
    GROUP BY snapshot_date
),
canonical_daily AS (
    SELECT
        r.snapshot_date,
        r.distinct_jobs_in_run AS canonical_distinct_jobs
    FROM qm_runs r
    JOIN daily_latest d
      ON r.snapshot_date = d.snapshot_date
     AND r.pulled_at = d.latest_pulled_at
)
SELECT
    rd.snapshot_date,
    rd.raw_rows_all_runs,
    cd.canonical_distinct_jobs
FROM raw_daily rd
JOIN canonical_daily cd
  ON rd.snapshot_date = cd.snapshot_date
WHERE rd.snapshot_date BETWEEN '2026-02-20' AND '2026-02-25'
ORDER BY rd.snapshot_date
"""

run_query(quantum_machines_sanity_query)


In [ ]:
multi_company_daily = daily_counts.copy()
company_count_by_date = multi_company_daily.groupby('snapshot_date')['company'].nunique()

if (company_count_by_date >= 2).any():
    multi_company_start_date = company_count_by_date[company_count_by_date >= 2].index.min()
else:
    multi_company_start_date = multi_company_daily['snapshot_date'].min()

multi_company_daily = multi_company_daily[multi_company_daily['snapshot_date'] >= multi_company_start_date]
pivot = multi_company_daily.pivot(index='snapshot_date', columns='company', values='job_count').sort_index()

# Reindex to a complete daily range and keep missing values as NaN.
# This prevents matplotlib from drawing misleading lines across large data gaps.
full_date_range = pd.date_range(start=pivot.index.min(), end=pivot.index.max(), freq='D')
pivot = pivot.reindex(full_date_range)
pivot.index.name = 'snapshot_date'

multi_company_start_date


In [ ]:
plt.figure(figsize=(12, 6))
for company in pivot.columns:
    plt.plot(pivot.index, pivot[company], marker='o', markersize=3, linewidth=1.5, label=company)

plt.title('Canonical Daily Jobs Over Time by Company')
plt.xlabel('Date')
plt.ylabel('Distinct Job Count (Latest Daily Snapshot)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
